In [ ]:
"""Scenario 4:  
The transactions folder contains 10 parquet files. Files part_0000 through part_0007 (8 million rows) have a column called 'amount'.
 Files part_0008 and part_0009 (2 million rows) — the latest batch from the payments team — have that same column renamed to 'transaction_amount'.
"""

#==>> Before — No Schema Handling (Two Failure Modes)
from pyspark.sql import SparkSession
from pyspark.sql.functions import broadcast
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .appName("Scenario_4")\
    .config("spark.sql.shuffle.partitions", "200") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
#transaction
transaction = spark.read.parquet(r"D:\data engineer\pyspark_practice\pyspark_scenarios\dataset1_transactions\transactions")

transaction.groupBy("customer_id").agg(sum("amount").alias("total_revenue"))

""" 
Failure Mode 1: Spark reads old files first then new files. New files do not have 'amount'. Spark silently fills those 2 million rows with NULL.
 Your aggregation runs but 20% of rows contribute Rs 0 to revenue. No error thrown. Finance team notices the 20% revenue drop 3 days later.

Failure Mode 2: Spark reads new files first then old files. Spark uses the new schema as reference and drops 'amount' entirely from the output.
 The column simply does not exist in the result. 8 million rows that should contribute revenue are silently excluded. No error thrown.
 
 """

DataFrame[customer_id: string, total_revenue: double]

In [ ]:
#After — mergeSchema with Null Handling
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .appName("Scenario_4")\
    .config("spark.sql.shuffle.partitions", "200") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
#transaction
transaction = spark.read.option("mergeSchema","True").parquet(r"D:\data engineer\pyspark_practice\pyspark_scenarios\dataset1_transactions\transactions")
# Both 'amount' and 'transaction_amount' columns now exist.
# Old files: amount=value, transaction_amount=NULL
# New files: amount=NULL, transaction_amount=value
transaction = transaction.withColumn("amount",coalesce(col("amount"),col("transaction_amount")))
transaction.groupBy("customer_id").agg(sum("amount").alias("total_revenue")).show()

"""Now Spark reads all files, unifies the schema taking the superset of all columns, and produces a single clean 'amount' column regardless of which
 original column name was used. All 10 million rows contribute correctly to revenue.
"""

+-----------+------------------+
|customer_id|     total_revenue|
+-----------+------------------+
|   C0220739|          153708.0|
|   C0051211|117303.26999999999|
|   C0145906|         264262.28|
|   C0153238|196940.69999999998|
|   C0037707|         130653.56|
|   C0125268|         130233.12|
|   C0011170|          289592.0|
|   C0149889|         135085.55|
|   C0066379|          90778.91|
|   C0154560|         110124.62|
|   C0106300|118381.89999999998|
|   C0041994|           76161.0|
|   C0138821|         126736.39|
|   C0255214|         160060.62|
|   C0087353|          65224.91|
|   C0150865|          95309.16|
|   C0161836|          99467.93|
|   C0095830|106666.56999999999|
|   C0288915|         174947.35|
|   C0170851| 65667.18999999999|
+-----------+------------------+
only showing top 20 rows

